## Imports e Configurações

In [1]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns

# Adiciona a pasta src ao path
sys.path.append(os.path.abspath(os.path.join('..')))

from src.dataset import get_svhn_loaders
from src.model import SVHNNet
from src.train_utils import train_one_epoch, evaluate

# Configuração de dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando device: {device}")

# Hiperparâmetros Fixos para comparação justa
BATCH_SIZE = 64
EPOCHS = 10  # SVHN converge rápido, 10 é suficiente para ver diferenças
LR_SGD = 0.01 # Learning Rate padrão para SGD
LR_ADAM = 0.001 # Adam geralmente precisa de LR menor

# Carregar Dados
train_loader, test_loader = get_svhn_loaders(batch_size=BATCH_SIZE)

Usando device: cpu
Using downloaded and verified file: ./data/train_32x32.mat
Using downloaded and verified file: ./data/test_32x32.mat


## Função de Experimento
Esta função é crucial. Ela garante que, para cada otimizador, você comece com um modelo "zerado" (pesos reinicializados). Se você não fizer isso, o segundo otimizador continuaria o treino do primeiro!

In [2]:
def run_experiment(optimizer_name, optimizer_params, model_class, train_loader, test_loader, epochs, track_gradients=False):
    """
    Cria um modelo novo, configura o otimizador e treina.
    Retorna históricos de loss, acurácia e opcionalmente normas de gradientes.
    """
    print(f"--- Iniciando Experimento: {optimizer_name} ---")
    
    # 1. Instanciar novo modelo (Resetar pesos)
    model = model_class().to(device)
    criterion = nn.CrossEntropyLoss()
    
    # 2. Configurar Otimizador com base no nome
    if optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), **optimizer_params)
    elif optimizer_name == 'SGD + Momentum':
        optimizer = optim.SGD(model.parameters(), **optimizer_params)
    elif optimizer_name == 'SGD + Nesterov':
        optimizer = optim.SGD(model.parameters(), **optimizer_params)
    elif optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), **optimizer_params)
    
    # 3. Loop de Treino
    train_losses = []
    val_accuracies = []
    grad_norms = [] if track_gradients else None
    
    for epoch in range(epochs):
        # Treinar uma época
        model.train()
        epoch_loss = 0.0
        epoch_grad_norms = []
        
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            
            # Capturar norma dos gradientes se solicitado
            if track_gradients:
                total_norm = 0.0
                for p in model.parameters():
                    if p.grad is not None:
                        param_norm = p.grad.data.norm(2)
                        total_norm += param_norm.item() ** 2
                total_norm = total_norm ** 0.5
                epoch_grad_norms.append(total_norm)
            
            optimizer.step()
            epoch_loss += loss.item()
        
        # Média da loss da época
        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)
        
        # Média das normas de gradientes da época
        if track_gradients:
            grad_norms.append(sum(epoch_grad_norms) / len(epoch_grad_norms))
        
        # Avaliar
        _, v_acc = evaluate(model, test_loader, criterion, device)
        val_accuracies.append(v_acc)
        
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Val Acc: {v_acc:.2f}%")
    
    if track_gradients:
        return train_losses, val_accuracies, grad_norms
    return train_losses, val_accuracies

## Execução dos Experimentos (SGD e Variantes)
Aqui cobrimos o Tópico 4 (SGD Simples, Momentum, Nesterov).

In [3]:
results = {}

# 1. SGD Vanilla (Puro)
# Geralmente é lento e oscila muito ou fica preso em platôs
losses, accs, grads = run_experiment(
    'SGD', 
    {'lr': LR_SGD}, 
    SVHNNet, train_loader, test_loader, EPOCHS, track_gradients=True
)
results['SGD'] = (losses, accs, grads)

# 2. SGD com Momentum
# O Momentum (geralmente 0.9) ajuda a acumular velocidade na direção certa
losses, accs, grads = run_experiment(
    'SGD + Momentum', 
    {'lr': LR_SGD, 'momentum': 0.9}, 
    SVHNNet, train_loader, test_loader, EPOCHS, track_gradients=True
)
results['SGD + Momentum'] = (losses, accs, grads)

# 3. SGD com Nesterov
# O Nesterov calcula o gradiente "à frente" da posição atual. Mais estável teoricamente.
losses, accs, grads = run_experiment(
    'SGD + Nesterov', 
    {'lr': LR_SGD, 'momentum': 0.9, 'nesterov': True}, 
    SVHNNet, train_loader, test_loader, EPOCHS, track_gradients=True
)
results['SGD + Nesterov'] = (losses, accs, grads)

--- Iniciando Experimento: SGD ---
Epoch 1/10 | Loss: 1.1831 | Val Acc: 79.60%
Epoch 2/10 | Loss: 0.5458 | Val Acc: 83.69%
Epoch 3/10 | Loss: 0.4501 | Val Acc: 86.29%
Epoch 4/10 | Loss: 0.3991 | Val Acc: 86.85%
Epoch 5/10 | Loss: 0.3620 | Val Acc: 88.55%
Epoch 6/10 | Loss: 0.3343 | Val Acc: 88.69%
Epoch 7/10 | Loss: 0.3104 | Val Acc: 89.03%
Epoch 8/10 | Loss: 0.2922 | Val Acc: 88.59%
Epoch 9/10 | Loss: 0.2752 | Val Acc: 89.29%
Epoch 10/10 | Loss: 0.2615 | Val Acc: 89.87%
--- Iniciando Experimento: SGD + Momentum ---
Epoch 1/10 | Loss: 0.6709 | Val Acc: 87.68%
Epoch 2/10 | Loss: 0.3559 | Val Acc: 88.34%
Epoch 3/10 | Loss: 0.2936 | Val Acc: 90.76%
Epoch 4/10 | Loss: 0.2516 | Val Acc: 90.19%
Epoch 5/10 | Loss: 0.2222 | Val Acc: 90.78%
Epoch 6/10 | Loss: 0.1991 | Val Acc: 91.49%
Epoch 7/10 | Loss: 0.1780 | Val Acc: 91.50%
Epoch 8/10 | Loss: 0.1583 | Val Acc: 91.48%
Epoch 9/10 | Loss: 0.1430 | Val Acc: 91.40%
Epoch 10/10 | Loss: 0.1299 | Val Acc: 91.20%
--- Iniciando Experimento: SGD + Nest

KeyboardInterrupt: 

## Execução do Adam
Aqui cobrimos o Tópico 2 (Adam). Note que usamos uma Learning Rate diferente (geralmente 1e-3), pois o Adam se comporta de forma distinta do SGD.

## Teoria: Como o Adam Funciona

O **Adam (Adaptive Moment Estimation)** é um otimizador que combina duas ideias poderosas:

### 1. **Momentum (Momento de Primeira Ordem)**
Mantém uma média móvel exponencial dos gradientes passados:

$$m_t = \beta_1 \cdot m_{t-1} + (1 - \beta_1) \cdot g_t$$

Onde:
- $m_t$ é o momento de primeira ordem (média dos gradientes)
- $g_t$ é o gradiente atual
- $\beta_1$ controla o "peso" dos gradientes passados (tipicamente 0.9)

### 2. **RMSProp/Escalonamento por Gradientes ao Quadrado (Momento de Segunda Ordem)**
Mantém uma média móvel dos gradientes ao quadrado para adaptar a taxa de aprendizado:

$$v_t = \beta_2 \cdot v_{t-1} + (1 - \beta_2) \cdot g_t^2$$

Onde:
- $v_t$ é o momento de segunda ordem (variância dos gradientes)
- $\beta_2$ controla a média móvel da variância (tipicamente 0.999)

### 3. **Correção de Viés (Bias Correction)**
Como $m_t$ e $v_t$ são inicializados em zero, eles ficam enviesados para zero nas primeiras iterações. A correção é:

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$$

$$\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

### 4. **Atualização Final dos Parâmetros**
$$\theta_{t+1} = \theta_t - \frac{\alpha}{\sqrt{\hat{v}_t} + \epsilon} \cdot \hat{m}_t$$

Onde:
- $\alpha$ é a taxa de aprendizado (learning rate)
- $\epsilon$ é uma constante pequena (tipicamente $10^{-8}$) para evitar divisão por zero

### Papel dos Hiperparâmetros:
- **$\beta_1$ (0.9)**: Controla a "memória" dos gradientes. Valores maiores = mais suavização
- **$\beta_2$ (0.999)**: Controla a adaptação da taxa de aprendizado. Valores maiores = mudanças mais graduais
- **$\epsilon$ ($10^{-8}$)**: Estabilidade numérica, evita divisão por zero quando gradientes são muito pequenos

## Demonstração Prática: Efeito dos Hiperparâmetros do Adam

Vamos experimentar com diferentes valores de β₁, β₂ e ϵ para demonstrar seu impacto no comportamento do otimizador.

In [ ]:
# Experimentos com Adam: variando hiperparâmetros
adam_experiments = {}

# Adam Padrão
adam_experiments['Adam (padrão)'] = run_experiment(
    'Adam', 
    {'lr': LR_ADAM, 'betas': (0.9, 0.999), 'eps': 1e-8},
    SVHNNet, train_loader, test_loader, EPOCHS, track_gradients=True
)

# β₁ menor (menos momentum) - converge de forma mais "errática"
adam_experiments['Adam (β₁=0.5)'] = run_experiment(
    'Adam', 
    {'lr': LR_ADAM, 'betas': (0.5, 0.999), 'eps': 1e-8},
    SVHNNet, train_loader, test_loader, EPOCHS, track_gradients=True
)

# β₂ menor (menos adaptação) - mais sensível a gradientes recentes
adam_experiments['Adam (β₂=0.9)'] = run_experiment(
    'Adam', 
    {'lr': LR_ADAM, 'betas': (0.9, 0.9), 'eps': 1e-8},
    SVHNNet, train_loader, test_loader, EPOCHS, track_gradients=True
)

# ϵ maior (mais estabilidade, menos adaptação)
adam_experiments['Adam (ϵ=1e-4)'] = run_experiment(
    'Adam', 
    {'lr': LR_ADAM, 'betas': (0.9, 0.999), 'eps': 1e-4},
    SVHNNet, train_loader, test_loader, EPOCHS, track_gradients=True
)

## Análise dos Hiperparâmetros do Adam

Visualização do impacto de diferentes valores de β₁, β₂ e ϵ:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Curvas de Loss para variações do Adam
ax1 = axes[0, 0]
for name, (losses, accs, grads) in adam_experiments.items():
    ax1.plot(losses, label=name, marker='o', markersize=4)
ax1.set_title('Impacto dos Hiperparâmetros - Curva de Loss', fontsize=12, fontweight='bold')
ax1.set_xlabel('Épocas')
ax1.set_ylabel('Loss (CrossEntropy)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Curvas de Acurácia
ax2 = axes[0, 1]
for name, (losses, accs, grads) in adam_experiments.items():
    ax2.plot(accs, label=name, marker='o', markersize=4)
ax2.set_title('Impacto dos Hiperparâmetros - Acurácia', fontsize=12, fontweight='bold')
ax2.set_xlabel('Épocas')
ax2.set_ylabel('Acurácia (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Curvas de Norma dos Gradientes
ax3 = axes[1, 0]
for name, (losses, accs, grads) in adam_experiments.items():
    ax3.plot(grads, label=name, marker='o', markersize=4)
ax3.set_title('Norma dos Gradientes por Época', fontsize=12, fontweight='bold')
ax3.set_xlabel('Épocas')
ax3.set_ylabel('Norma L2 dos Gradientes')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Comparação final (Loss x Acurácia)
ax4 = axes[1, 1]
for name, (losses, accs, grads) in adam_experiments.items():
    ax4.scatter(losses[-1], accs[-1], s=150, label=name, alpha=0.7)
ax4.set_title('Loss Final vs Acurácia Final', fontsize=12, fontweight='bold')
ax4.set_xlabel('Loss Final')
ax4.set_ylabel('Acurácia Final (%)')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Tabela resumo
print("\n" + "="*80)
print("RESUMO DOS EXPERIMENTOS COM ADAM")
print("="*80)
for name, (losses, accs, grads) in adam_experiments.items():
    print(f"{name:25} | Loss Final: {losses[-1]:.4f} | Acurácia Final: {accs[-1]:.2f}% | Grad Norm Final: {grads[-1]:.4f}")
print("="*80)

In [ ]:
# 4. Adam
# Combina Momentum + RMSProp (escala adaptativa).
# Geralmente o campeão de convergência inicial.
losses, accs, grads = run_experiment(
    'Adam', 
    {'lr': LR_ADAM}, 
    SVHNNet, train_loader, test_loader, EPOCHS, track_gradients=True
)
results['Adam'] = (losses, accs, grads)

## Visualização Comparativa e Análise
Gera os gráficos obrigatórios para a Nota Técnica.

In [ ]:
# Configuração dos plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Curva de Loss (Treino) - "Velocidade de Convergência"
ax1 = axes[0, 0]
for name, (losses, accs, grads) in results.items():
    ax1.plot(losses, label=name, marker='.')
ax1.set_title('Comparação: Curva de Perda (Loss)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Épocas')
ax1.set_ylabel('Loss (CrossEntropy)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Acurácia de Validação - "Desempenho Final"
ax2 = axes[0, 1]
for name, (losses, accs, grads) in results.items():
    ax2.plot(accs, label=name, marker='.')
ax2.set_title('Comparação: Acurácia de Validação', fontsize=12, fontweight='bold')
ax2.set_xlabel('Épocas')
ax2.set_ylabel('Acurácia (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Norma dos Gradientes - "Estabilidade do Treinamento"
ax3 = axes[1, 0]
for name, (losses, accs, grads) in results.items():
    ax3.plot(grads, label=name, marker='.')
ax3.set_title('Comparação: Norma dos Gradientes', fontsize=12, fontweight='bold')
ax3.set_xlabel('Épocas')
ax3.set_ylabel('Norma L2 dos Gradientes')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Comparação SGD vs Adam (Destaque)
ax4 = axes[1, 1]
sgd_losses, sgd_accs, sgd_grads = results['SGD']
adam_losses, adam_accs, adam_grads = results['Adam']

ax4_twin = ax4.twinx()
line1 = ax4.plot(sgd_losses, 'b-', label='SGD Loss', linewidth=2, marker='o')
line2 = ax4.plot(adam_losses, 'r-', label='Adam Loss', linewidth=2, marker='s')
line3 = ax4_twin.plot(sgd_grads, 'b--', label='SGD Grad Norm', linewidth=2, alpha=0.6)
line4 = ax4_twin.plot(adam_grads, 'r--', label='Adam Grad Norm', linewidth=2, alpha=0.6)

ax4.set_title('SGD vs Adam: Loss e Gradientes', fontsize=12, fontweight='bold')
ax4.set_xlabel('Épocas')
ax4.set_ylabel('Loss', color='black')
ax4_twin.set_ylabel('Norma dos Gradientes', color='gray')

# Combinar legendas
lines = line1 + line2 + line3 + line4
labels = [l.get_label() for l in lines]
ax4.legend(lines, labels, loc='upper right')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Tabela de resultados finais
print("\n" + "="*80)
print("RESUMO COMPARATIVO FINAL")
print("="*80)
for name, (losses, accs, grads) in results.items():
    print(f"{name:20} | Loss Final: {losses[-1]:.4f} | Acurácia: {accs[-1]:.2f}% | Grad Norm: {grads[-1]:.4f}")
print("="*80)

## Interpretação dos Resultados

### O que observar nas curvas:

**Curva de Loss:**
- **SGD puro**: Tende a ser mais lento e oscilatório
- **SGD + Momentum/Nesterov**: Convergência mais suave
- **Adam**: Convergência inicial rápida devido à adaptação automática da taxa de aprendizado

**Curva de Acurácia:**
- Compare a velocidade de convergência (quantas épocas para atingir boa acurácia)
- Observe a estabilidade (oscilações vs. curva suave)

**Norma dos Gradientes:**
- **Alta no início**: O modelo está longe do mínimo, gradientes grandes
- **Diminuição gradual**: Aproximação do mínimo local
- **Adam**: Geralmente mostra gradientes mais estáveis devido ao escalonamento adaptativo
- **SGD**: Pode mostrar mais variabilidade

### Por que Adam funciona bem:

1. **Momentum ($\beta_1$)**: Acumula direção consistente, acelera em vales
2. **Adaptação ($\beta_2$)**: Ajusta taxa de aprendizado por parâmetro, útil em gradientes com escalas diferentes
3. **Correção de viés**: Garante início correto do treinamento
4. **$\epsilon$**: Evita divisão por zero quando gradientes são muito pequenos

### Quando usar cada otimizador:

- **SGD simples**: Baseline, debugging, quando você quer controle total
- **SGD + Momentum**: Bom equilíbrio, funciona bem com schedule de learning rate
- **Adam**: Excelente ponto de partida, funciona "out of the box" em muitos problemas
- **Escolha final**: Depende do problema! Teste empiricamente.